In [14]:
import open3d as o3d
import numpy as np
import os
import lib3mf
from lib3mf_common import *

model_name = "pi 3mm side paint"
model_path = f'input_models/{model_name}.3mf'
stl_path = f'input_models/{model_name}.stl'

if os.path.isfile(model_path):
    print("Found file!")
else:
    print("Did not find file!")
wrapper = get_wrapper()
model = wrapper.CreateModel()
read_3mf_file_to_model(model, model_path)

# convert to STL
writer = model.QueryWriter("stl")
print(f"Writing {stl_path}...")
writer.WriteToFile(stl_path)
print("Done")

print("Testing mesh in Open3D...")
mesh = o3d.io.read_triangle_mesh(stl_path)
print(mesh)
print('Vertices:')
print(np.asarray(mesh.vertices))
print('Triangles:')
print(np.asarray(mesh.triangles))

Found file!
Writing input_models/pi 3mm side paint.stl...
Done
Testing mesh in Open3D...
TriangleMesh with 2610 points and 870 triangles.
Vertices:
[[ 25.          -5.          10.        ]
 [ 22.5         -5.          10.        ]
 [ 25.          -2.5         10.        ]
 ...
 [-22.05882263   5.          17.1428566 ]
 [-19.11764526   5.          14.28571415]
 [-22.05882263   5.          14.28571415]]
Triangles:
[[   0    1    2]
 [   3    4    5]
 [   6    7    8]
 ...
 [2601 2602 2603]
 [2604 2605 2606]
 [2607 2608 2609]]


In [15]:
print("Try to render a mesh with normals (exist: " +
      str(mesh.has_vertex_normals()) + ") and colors (exist: " +
      str(mesh.has_vertex_colors()) + ")")
o3d.visualization.draw_geometries([mesh])
print("A mesh with no normals and no colors does not look good.")

Try to render a mesh with normals (exist: True) and colors (exist: False)
A mesh with no normals and no colors does not look good.


In [40]:
import networkx as nx
import numpy as np
import pyvista as pv
import tetgen
from scipy.optimize import minimize, least_squares
from scipy.spatial.transform import Rotation as R
import open3d as o3d
import time
import pickle
import base64
import stl
from stl import mesh
import os
import lib3mf
from lib3mf_common import *
import copy

def process_mesh(model_mesh):
    def encode_object(obj):
        return base64.b64encode(pickle.dumps(obj)).decode('utf-8')
    
    up_vector = np.array([0, 0, 1])

    def decode_object(encoded_str):
        return pickle.loads(base64.b64decode(encoded_str))

    # process mesh as in S4_Slicer and return extracted surface

    input_tet = tetgen.TetGen(np.asarray(model_mesh.vertices), np.asarray(model_mesh.triangles))
    input_tet.tetrahedralize()
    input_tet = input_tet.grid

    PART_OFFSET = np.array([0., 0., 0.])
    x_min, x_max, y_min, y_max, z_min, z_max = input_tet.bounds
    input_tet.points -= np.array([(x_min + x_max) / 2, (y_min + y_max) / 2, z_min]) + PART_OFFSET


    # find neighbours
    cell_neighbour_dict = {neighbour_type: {face: [] for face in range(input_tet.number_of_cells)} for neighbour_type in ["point", "edge", "face"]}
    for neighbour_type in ["point", "edge", "face"]:
        cell_neighbours = []
        for cell_index in range(input_tet.number_of_cells):
            neighbours = input_tet.cell_neighbors(cell_index, f"{neighbour_type}s")
            for neighbour in neighbours:
                if neighbour > cell_index:
                    cell_neighbours.append((cell_index, neighbour))
        for face_1, face_2 in np.array(cell_neighbours):
            cell_neighbour_dict[neighbour_type][face_1].append(face_2)
            cell_neighbour_dict[neighbour_type][face_2].append(face_1)

        input_tet.field_data[f"cell_{neighbour_type}_neighbours"] = np.array(cell_neighbours)

    cell_neighbour_graph = nx.Graph()
    cell_centers = input_tet.cell_centers().points
    for edge in input_tet.field_data["cell_point_neighbours"]: # use point neighbours for best accuracy
        distance = np.linalg.norm(cell_centers[edge[0]] - cell_centers[edge[1]])
        cell_neighbour_graph.add_weighted_edges_from([(edge[0], edge[1], distance)])

    def update_tet_attributes(tet):
        '''
        Calculate face normals, face centers, cell centers, and overhang angles for each cell in the tetrahedral mesh.
        '''

        surface_mesh = tet.extract_surface()
        cell_to_face = decode_object(tet.field_data["cell_to_face"])

        # put general data in field_data for easy access
        cells = tet.cells.reshape(-1, 5)[:, 1:] # assume all cells have 4 vertices
        tet.add_field_data(cells, "cells")
        cell_vertices = tet.points
        tet.add_field_data(cell_vertices, "cell_vertices")
        faces = surface_mesh.faces.reshape(-1, 4)[:, 1:] # assume all faces have 3 vertices
        tet.add_field_data(faces, "faces")
        face_vertices = surface_mesh.points
        tet.add_field_data(face_vertices, "face_vertices")

        tet.cell_data['face_normal'] = np.full((tet.number_of_cells, 3), np.nan)
        surface_mesh_face_normals = surface_mesh.face_normals
        for cell_index, face_indices in cell_to_face.items():
            face_normals = surface_mesh_face_normals[face_indices]
            # get the normal facing the most down
            most_down_normal_index = np.argmin(face_normals[:, 2])
            tet.cell_data['face_normal'][cell_index] = face_normals[most_down_normal_index]
        tet.cell_data['face_normal'] =  tet.cell_data['face_normal'] / np.linalg.norm(tet.cell_data['face_normal'], axis=1)[:, None]

        tet.cell_data['face_center'] = np.empty((tet.number_of_cells, 3))
        tet.cell_data['face_center'][:,:] = np.nan
        surface_mesh_cell_centers = surface_mesh.cell_centers().points
        for cell_index, face_indices in cell_to_face.items():
            face_centers = surface_mesh_cell_centers[face_indices]
            # get the normal facing the most down
            most_down_center_index = np.argmin(face_centers[:, 2])
            tet.cell_data['face_center'][cell_index] = face_centers[most_down_center_index]

        tet.cell_data["cell_center"] = tet.cell_centers().points

        # calculate bottom cells
        bottom_cell_threshold = np.nanmin(tet.cell_data['face_center'][:, 2])+0.3
        bottom_cells_mask = tet.cell_data['face_center'][:, 2] < bottom_cell_threshold
        tet.cell_data['is_bottom'] = bottom_cells_mask
        bottom_cells = np.where(bottom_cells_mask)[0]

        face_normals = tet.cell_data['face_normal'].copy()
        face_normals[bottom_cells_mask] = np.nan # make bottom faces not angled
        overhang_angle = np.arccos(np.dot(face_normals, up_vector))
        tet.cell_data['overhang_angle'] = overhang_angle

        overhang_direction = face_normals[:, :2].copy()
        overhang_direction /= np.linalg.norm(overhang_direction, axis=1)[:, None]
        tet.cell_data['overhang_direction'] = overhang_direction

        # calculate if cell will print in air by seeing if any cell centers along path to base are higher
        IN_AIR_THRESHOLD = 1
        tet.cell_data['in_air'] = np.full(tet.number_of_cells, False)

        _, paths_to_bottom = nx.multi_source_dijkstra(cell_neighbour_graph, set(bottom_cells))

        # put it in cell data
        tet.cell_data['path_to_bottom'] = np.full((tet.number_of_cells, np.max([len(x) for x in paths_to_bottom.values()])), -1)
        for cell_index, path_to_bottom in paths_to_bottom.items():
            tet.cell_data['path_to_bottom'][cell_index, :len(path_to_bottom)] = path_to_bottom

        # calculate if cell is in air
        for cell_index in range(tet.number_of_cells):
            path_to_bottom = paths_to_bottom[cell_index]
            if len(path_to_bottom) > 1:
                cell_heights = tet.cell_data['cell_center'][path_to_bottom, 2]
                if np.any(cell_heights > tet.cell_data['cell_center'][cell_index, 2] + IN_AIR_THRESHOLD):
                    tet.cell_data['in_air'][cell_index] = True

        return tet

    def calculate_tet_attributes(tet):
        '''
        Calculate shared vertices between cells, cell to face & face to cell relations, and bottom cells of the tetrahedral mesh.
        '''

        surface_mesh = tet.extract_surface()

        # put general data in field_data for easy access
        cells = tet.cells.reshape(-1, 5)[:, 1:] # assume all cells have 4 vertices
        tet.add_field_data(cells, "cells")
        cell_vertices = tet.points
        tet.add_field_data(cell_vertices, "cell_vertices")
        faces = surface_mesh.faces.reshape(-1, 4)[:, 1:] # assume all faces have 3 vertices
        tet.add_field_data(faces, "faces")
        face_vertices = surface_mesh.points
        tet.add_field_data(face_vertices, "face_vertices")

        # calculate shared vertices
        shared_vertices = []
        for cell_1, cell_2 in tet.field_data["cell_point_neighbours"]:
            shared_vertices_these_faces = np.intersect1d(cells[cell_1], cells[cell_2])
            for vertex in shared_vertices_these_faces:
                shared_vertices.append({
                        "cell_1_index": cell_1,
                        "cell_2_index": cell_2,
                        "cell_1_vertex_index": np.where(cells[cell_1] == vertex)[0][0],
                        "cell_2_vertex_index": np.where(cells[cell_2] == vertex)[0][0],
                    })

        # calculate cell to face & face to cell relations
        cell_to_face = {}
        face_to_cell = {face_index: [] for face_index in range(len(faces))}
        cell_to_face_vertices = {}
        face_to_cell_vertices = {}
        for cell_vertex_index, cell_vertex in enumerate(tet.field_data["cell_vertices"].reshape(-1, 3)):
            face_vertex_index = np.where((face_vertices == cell_vertex).all(axis=1))[0]
            if len(face_vertex_index) == 1:
                cell_to_face_vertices[cell_vertex_index] = face_vertex_index[0]
                face_to_cell_vertices[face_vertex_index[0]] = cell_vertex_index

        for cell_index, cell in enumerate(tet.field_data["cells"]):
            face_vertex_indices = [cell_to_face_vertices[cell_vertex_index] for cell_vertex_index in cell if cell_vertex_index in cell_to_face_vertices]
            if len(face_vertex_indices) >= 3:
                extracted = surface_mesh.extract_points(face_vertex_indices, adjacent_cells=False)
                if extracted.number_of_cells >= 1:
                    cell_to_face[cell_index] = list(extracted.cell_data['vtkOriginalCellIds'])
                    for face_index in extracted.cell_data['vtkOriginalCellIds']:
                        face_to_cell[face_index].append(cell_index)

        tet.add_field_data(encode_object(cell_to_face), "cell_to_face")
        tet.add_field_data(encode_object(face_to_cell), "face_to_cell")

        # calculate has_face attribute
        tet.cell_data['has_face'] = np.zeros(tet.number_of_cells)
        for cell_index, face_indices in cell_to_face.items():
            tet.cell_data['has_face'][cell_index] = 1

        tet = update_tet_attributes(tet)

        # calculate bottom cells
        bottom_cells_mask = tet.cell_data['is_bottom']
        bottom_cells = np.where(bottom_cells_mask)[0]

        tet.cell_data['overhang_angle'][bottom_cells] = np.nan

        return tet, bottom_cells_mask, bottom_cells


    bottom_cells_mask = None
    bottom_cells = None
    input_tet, bottom_cells_mask, bottom_cells = calculate_tet_attributes(input_tet)

    # find bottom cell groups that are connected
    bottom_cell_graph = nx.Graph()
    for cell_index in bottom_cells:
        bottom_cell_graph.add_node(cell_index)
    cell_point_neighbour_dict = cell_neighbour_dict["point"]
    for cell_index in bottom_cells:
        for neighbour in cell_point_neighbour_dict[cell_index]:
            if neighbour in bottom_cells:
                bottom_cell_graph.add_edge(cell_index, neighbour)

    bottom_cell_groups = [list(x) for x in list(nx.connected_components(bottom_cell_graph))]

    undeformed_tet = input_tet.copy()

    surface = input_tet.extract_surface() #pyvista
    return surface

# from https://www.open3d.org/docs/0.12.0/tutorial/pipelines/global_registration.html with edits
def tri_to_processed_pt_cld(tri_mesh, voxel_size):
    pcd = tri_mesh.sample_points_uniformly(500)

    print(":: Downsample with a voxel size %.3f." % voxel_size)
    pcd_down = pcd.voxel_down_sample(voxel_size)

    radius_normal = voxel_size * 2
    print(":: Estimate normal with search radius %.3f." % radius_normal)
    pcd_down.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=30))

    radius_feature = voxel_size * 5
    print(":: Compute FPFH feature with search radius %.3f." % radius_feature)
    pcd_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down,
        o3d.geometry.KDTreeSearchParamHybrid(radius=radius_feature, max_nn=100))
    return pcd_down, pcd_fpfh
    
# from https://www.open3d.org/docs/0.12.0/tutorial/pipelines/global_registration.html
def execute_global_registration(source_down, target_down, source_fpfh,
                                target_fpfh, voxel_size):
    distance_threshold = voxel_size * 1.5
    print(":: RANSAC registration on downsampled point clouds.")
    print("   Since the downsampling voxel size is %.3f," % voxel_size)
    print("   we use a liberal distance threshold %.3f." % distance_threshold)
    result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        source_down, target_down, source_fpfh, target_fpfh, True,
        distance_threshold,
        o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
        3, [
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(
                0.9),
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(
                distance_threshold)
        ], o3d.pipelines.registration.RANSACConvergenceCriteria(100000, 0.999))
    return result

# from https://www.open3d.org/docs/0.12.0/tutorial/pipelines/global_registration.html
def execute_fast_global_registration(source_down, target_down, source_fpfh,
                                     target_fpfh, voxel_size):
    distance_threshold = voxel_size * 0.5
    print(":: Apply fast global registration with distance threshold %.3f" \
            % distance_threshold)
    result = o3d.pipelines.registration.registration_fast_based_on_feature_matching(
        source_down, target_down, source_fpfh, target_fpfh,
        o3d.pipelines.registration.FastGlobalRegistrationOption(
            maximum_correspondence_distance=distance_threshold))
    return result

def local_registration():
    return

# from global registration tutorial: https://www.open3d.org/docs/0.12.0/tutorial/pipelines/global_registration.html
def draw_registration_result(source, target, transformation):
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    source_temp.paint_uniform_color([1, 0.706, 0])
    target_temp.paint_uniform_color([0, 0.651, 0.929])
    source_temp.transform(transformation)
    o3d.visualization.draw_geometries([source_temp, target_temp],
                                      zoom=0.4559,
                                      front=[0.6452, -0.3036, -0.7011],
                                      lookat=[1.9892, 2.0208, 1.8945],
                                      up=[-0.2779, -0.9482, 0.1556])

# expects to be run from S4_Slicer folder
# Load 3MF
model_name = "z mount 3mm"
model_path = f'input_models/{model_name}.3mf'
stl_path = f'input_models/{model_name}.stl'

if os.path.isfile(model_path):
    print("Found file!")
else:
    print("Did not find file!")
    sys.exit()
wrapper = get_wrapper()
model = wrapper.CreateModel()
read_3mf_file_to_model(model, model_path)

# convert to STL
writer = model.QueryWriter("stl")
print(f"Writing {stl_path}...")
writer.WriteToFile(stl_path)
print("Done")

mf_mesh = o3d.io.read_triangle_mesh(stl_path)
# get o3d triangle mesh from pyvista PolyData containing vertices, triangles
# we do this to allow downsampling in case we use large meshes
disordered_mesh = process_mesh(mf_mesh)
o3d_vert = o3d.utility.Vector3dVector(disordered_mesh.points)
o3d_tri = o3d.utility.Vector3iVector(disordered_mesh.faces.reshape(-1, 4)[:, 1:])
disordered_mesh = o3d.geometry.TriangleMesh(o3d_vert, o3d_tri)

# draw to check scene
#draw_registration_result(mf_mesh, disordered_mesh, np.eye(4))

# check disordered mesh vs 3mf vertex list
mf_mesh = model.GetMeshObjectByID(1)
mf_verts = []
for i in range(mf_mesh.GetVertexCount()):
    vert = [x for x in mf_mesh.GetVertex(i).Coordinates]
    mf_verts.append(vert)
mf_verts = np.asarray(mf_verts)

mf_tri = []
for i in range(mf_mesh.GetTriangleCount()):
    tri = [y for y in mf_mesh.GetTriangle(i).Indices]
    mf_tri.append(tri)
mf_tri = np.asarray(mf_tri)
mf_verts = o3d.utility.Vector3dVector(mf_verts)
mf_tri = o3d.utility.Vector3iVector(mf_tri)
mf_mesh = o3d.geometry.TriangleMesh(mf_verts, mf_tri)
# draw to check scene
#draw_registration_result(mf_mesh, disordered_mesh, np.eye(4))

# generate downsampled point cloud on both for RANSAC
voxel_size = 0.01
mf_cld, mf_fpfh = tri_to_processed_pt_cld(mf_mesh, voxel_size)
disordered_cld, disordered_fpfh = tri_to_processed_pt_cld(disordered_mesh,voxel_size)
draw_registration_result(mf_cld, disordered_cld, np.eye(4))

# run RANSAC for global registration
result_ransac = execute_fast_global_registration(source_down=mf_cld, target_down=disordered_cld,
                                            source_fpfh=mf_fpfh, target_fpfh=disordered_fpfh,
                                            voxel_size=voxel_size)
print(result_ransac)
print(result_ransac.transformation)
draw_registration_result(mf_cld, disordered_cld, result_ransac.transformation)

Found file!
Writing input_models/z mount 3mm.stl...
Done


/tmp/ipykernel_7398/2214790498.py:109: RuntimeWarning: invalid value encountered in divide
  overhang_direction /= np.linalg.norm(overhang_direction, axis=1)[:, None]


:: Downsample with a voxel size 0.010.
:: Estimate normal with search radius 0.020.
:: Compute FPFH feature with search radius 0.050.
:: Downsample with a voxel size 0.010.
:: Estimate normal with search radius 0.020.
:: Compute FPFH feature with search radius 0.050.
:: Apply fast global registration with distance threshold 0.005
RegistrationResult with fitness=0.000000e+00, inlier_rmse=0.000000e+00, and correspondence_set size of 0
Access transformation to get result.
[[ 1.         -0.          0.          0.02588417]
 [-0.          1.         -0.         -0.09770484]
 [ 0.         -0.          1.         14.15946814]
 [-0.          0.         -0.          1.        ]]


In [ ]:
# cannot find transformation in build items
print(model.GetBuildItems())
print(model.GetBuildItems().Count())
iter = model.GetBuildItems()
# print(iter.GetCurrent())
# print(model.GetBuildItems().GetCurrent().GetObjectTransform())

# can find in components?
print("Components")
print(model.GetComponentsObjects())
# print(model.GetComponentsObjects().GetCurrentComponentsObject())
# print(model.GetComponentsObjectByID(1))
# print(model.GetComponentsObjectByID(1).GetComponent(1))

1
Components


ELib3MFException: Lib3MFException 102: the current index of an iterator is invalid